# Lekcija 10: Konvolucija

Konvolucija je temeljna operacija za zamućivanje, izoštravanje i detekciju rubova &mdash; ujedno je i ključna operacija konvolucijskih slojeva konvolucijske neuronske mreže (Lekcija 34). U ovoj ćemo je lekciji implementirati od početka, razjasniti čestu nedoumicu između konvolucije i korelacije te prikazati nekoliko klasičnih filtara.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Jednodimenzionalna konvolucija

Prije obrade dvodimenzionalnih slika počnimo s jednostavnijim problemom konvolucije dvaju jednodimenzionalnih polja. Za signal $I$ i jezgru $K$, diskretna konvolucija jest

$$(I * K)(x) = \sum_{i} K(i)\, I(x - i)$$

Konkretno: **obrnite** redoslijed elemenata jezgre, **pomičite** je duž signala za jedno mjesto te na svakom položaju izračunajte skalarni produkt obrnute jezgre i dijela signala s kojim se preklapa. Isprobajmo to na signalu koji prvo raste, a zatim pada, uz jednostavnu jezgru s 3 koeficijenta $[-1, 0, 1]$ &mdash; diskretnu derivaciju koja računa nagib.


In [ ]:
def convolve1d(signal, kernel):
    ksize = len(kernel)
    pad = ksize // 2
    padded = np.pad(signal, pad, mode='constant')
    flipped = kernel[::-1]
    out = np.zeros_like(signal, dtype=np.float64)
    for n in range(len(signal)):
        window = padded[n:n + ksize]
        out[n] = np.dot(window, flipped)
    return out

signal = np.array([1, 2, 3, 4, 5, 4, 3, 2, 1], dtype=np.float64)
kernel = 0.5 * np.array([1, 0, -1], dtype=np.float64) # pre-flipped; scaling factor ensures that result is slope

mine = convolve1d(signal, kernel)
reference = np.convolve(signal, kernel, mode='same')
print('signal:   ', signal)
print('mine:     ', mine)
print('np.convolve:', reference)
print('match:', np.allclose(mine, reference))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot(signal, marker='o')
axes[0].set_title('Signal')
axes[1].plot(mine, marker='o', color='tab:orange')
axes[1].axhline(0, color='gray', linewidth=0.8)
axes[1].set_title('Signal * [1, 0, -1]')
plt.tight_layout()
plt.show()

Primijetite da je izlaz pozitivan dok signal raste, jednak nuli na vrhu i negativan dok signal pada &mdash; upravo ono što bi jezgra nalik derivaciji trebala dati.


## Dvodimenzionalna konvolucija

Dvodimenzionalna inačica konvolucije, koja se primjenjuje na slike, koristi istu ideju obrtanja i pomicanja kao prethodni jednodimenzionalni slučaj, samo duž dviju osi:

$$(I * K)(x, y) = \sum_{i}\sum_{j} K(i, j)\, I(x - i,\, y - j)$$

gdje je $I$ slika, a $K$ jezgra. Kao i u jednodimenzionalnoj inačici, ključni detalj su predznaci *minus*: jezgra se obrće, odnosno rotira za 180°, prije pomicanja po slici. To je važno kada jezgra nije simetrična.

**Korelacija** koristi istu ideju, ali bez obrtanja jezgre:

$$(I \star K)(x, y) = \sum_{i}\sum_{j} K(i, j)\, I(x + i,\, y + j)$$

U praksi većina biblioteka za obradu slika &mdash; uključujući funkciju `cv2.filter2D` iz biblioteke OpenCV &mdash; zapravo implementira *korelaciju*, a ne konvoluciju, iako se često neformalno govori o „konvoluciji s jezgrom”. Za simetrične jezgre, poput jezgre filtra srednje vrijednosti ili Gaussove jezgre, rezultati su identični, pa ta razlika u praksi često nije važna. Ipak, korisno je znati da postoji.


## Implementacija konvolucije od početka

Za implementaciju dvodimenzionalne konvolucije obrnite jezgru i vodoravno i okomito, proširite sliku rubnim pikselima kako jezgra nikada ne bi izašla izvan nje te na svakom položaju pomnožite elemente obrnute jezgre s odgovarajućim pikselima ispod nje i zbrojite rezultate. Kod u nastavku izravno implementira taj postupak &mdash; uz jednu razliku radi brzine: petlja prolazi kroz malobrojne elemente jezgre umjesto kroz brojne piksele slike, pa za svaku težinu jezgre odjednom pomiče i pribraja cijelu proširenu sliku.


In [ ]:
def convolve2d(image, kernel, border=cv2.BORDER_REFLECT101):
    """True convolution (kernel is flipped), single-channel, float output."""
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2
    flipped = kernel[::-1, ::-1]
    padded = cv2.copyMakeBorder(image, pad_h, pad_h, pad_w, pad_w, border)

    out = np.zeros(image.shape, dtype=np.float64)
    for i in range(kh):  # loop over kernel values for speed (kernel is smaller than image)
        for j in range(kw):
            out += flipped[i, j] * padded[i:i + image.shape[0], j:j + image.shape[1]].astype(np.float64)
    return out

### Provjera usporedbom s bibliotekom OpenCV

Budući da funkcija `cv2.filter2D` izračunava korelaciju, predajemo joj *prethodno obrnutu* jezgru kako bismo dobili rezultat stvarne konvolucije za usporedbu.


In [ ]:
test_img = np.zeros((20, 20), dtype=np.uint8)
test_img[3:17, 3:6] = 200   # a small "corner" shape (no symmetry), so flipping is visible
test_img[3:6, 3:14] = 200

asymmetric_kernel = np.array([[1, 2, -1],
                               [0, 1,  3],
                               [-2, 1, 0]], dtype=np.float64)

mine = convolve2d(test_img, asymmetric_kernel)
reference = cv2.filter2D(test_img, cv2.CV_64F, asymmetric_kernel[::-1, ::-1],
                          borderType=cv2.BORDER_REFLECT101)

print('max abs difference:', np.abs(mine - reference).max())

### Vizualna usporedba konvolucije i korelacije

Za simetričnu jezgru konvolucija i korelacija daju *identične* rezultate. Za asimetričnu jezgru rezultati su različiti &mdash; ali mnoge asimetrične jezgre koje se koriste u praksi uzrokuju jednostavnu promjenu predznaka.


In [ ]:
correlated = cv2.filter2D(test_img, cv2.CV_64F, asymmetric_kernel, borderType=cv2.BORDER_REFLECT101)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [test_img, mine, correlated],
                          ['Original', 'Convolution\n(kernel flipped)', 'Correlation\n(kernel not flipped)']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

print('convolution == correlation with kernel rotated 180?',
      np.allclose(mine, cv2.filter2D(test_img, cv2.CV_64F, asymmetric_kernel[::-1, ::-1],
                                      borderType=cv2.BORDER_REFLECT101)))

## Obrada rubova slike

U blizini rubova jezgra djelomično izlazi izvan slike. Funkcija `convolve2d` prvo koristi `cv2.copyMakeBorder` za proširenje slike; način proširenja utječe na rezultat uz rub. Uobičajene mogućnosti su proširenje konstantom, primjerice nulama, ponavljanje rubnog piksela i zrcaljenje preko ruba (`REFLECT101` izbjegava ponavljanje samog rubnog piksela). U ovom primjeru unutarnje polje veličine $4 \times 4$ svakog izlaza identično je izvornom polju &mdash; razlikuje se samo vanjski prsten.


In [ ]:
small = np.arange(1, 17, dtype=np.float64).reshape(4, 4)

for name, mode in [('CONSTANT (zero)', cv2.BORDER_CONSTANT),
                    ('REPLICATE', cv2.BORDER_REPLICATE),
                    ('REFLECT101', cv2.BORDER_REFLECT101)]:
    padded = cv2.copyMakeBorder(small, 1, 1, 1, 1, mode)
    print(f'{name}:\n{padded}\n')

## Klasični filtri prikazani jezgrama

Kada znamo izračunati konvoluciju, većina filtara svodi se na odabir jezgre:

- **Filtar srednje vrijednosti (engl. box blur)**: svi susjedi imaju jednake težine &mdash; usrednjavanjem smanjuje šum, ali zamućuje rubove.
- **Gaussovo zamućivanje**: susjedi su ponderirani prema zvonolikoj krivulji &mdash; težine postupnije opadaju, uz manje pravokutnih artefakata nego kod filtra srednje vrijednosti.
- **Izoštravanje**: pojačava središnji piksel u odnosu na njegove susjede.
- **Sobelov filtar (detekcija rubova)**: asimetrična jezgra koja snažno reagira na promjene intenziteta u jednom smjeru &mdash; ovdje je razlika između konvolucije i korelacije važna jer obrtanje asimetrične jezgre mijenja predznak rezultata.


In [ ]:
photo_like = np.zeros((100, 100), dtype=np.uint8)
cv2.rectangle(photo_like, (20, 20), (80, 80), 200, -1)
cv2.circle(photo_like, (50, 50), 20, 60, -1)
rng = np.random.default_rng(0)
photo_like = photo_like.astype(np.float64) + rng.normal(0, 12, photo_like.shape)
photo_like = np.clip(photo_like, 0, 255)

box = np.ones((5, 5)) / 25

gx, gy = np.meshgrid(np.arange(5) - 2, np.arange(5) - 2)
sigma = 1.0
gaussian = np.exp(-(gx**2 + gy**2) / (2 * sigma**2))
gaussian /= gaussian.sum()

sharpen = np.array([[0, -1, 0],
                     [-1, 5, -1],
                     [0, -1, 0]], dtype=np.float64)

sobel_x = np.array([[-1, 0, 1],
                     [-2, 0, 2],
                     [-1, 0, 1]], dtype=np.float64)

results = {
    'Original (noisy)': photo_like,
    'Box blur': convolve2d(photo_like, box),
    'Gaussian blur': convolve2d(photo_like, gaussian),
    'Sharpen': convolve2d(photo_like, sharpen),
    'Sobel x (edges)': convolve2d(photo_like, sobel_x),
}

fig, axes = plt.subplots(1, 5, figsize=(15, 3.5))
for ax, (title, im) in zip(axes, results.items()):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Separabilne jezgre: postupak za ubrzavanje

Prethodna Gaussova jezgra je **separabilna**: može se zapisati kao vanjski produkt dviju jednodimenzionalnih jezgri, $K = k_x \, k_y^\top$. To znači da je konvolucija s punom jezgrom veličine $n \times n$ ekvivalentna konvoluciji s jezgrom veličine $1 \times n$, a zatim s jezgrom veličine $n \times 1$ &mdash; čime se broj operacija po pikselu smanjuje s $O(n^2)$ na $O(2n)$.


In [ ]:
k1d = np.exp(-(np.arange(5) - 2) ** 2 / (2 * sigma**2))
k1d /= k1d.sum()

outer_product = np.outer(k1d, k1d)
print('2D kernel == outer product of 1D kernels?', np.allclose(gaussian, outer_product))

separable_result = convolve2d(convolve2d(photo_like, k1d.reshape(1, -1)), k1d.reshape(-1, 1))
full_result = convolve2d(photo_like, gaussian)

print('separable == full 2D convolution?', np.allclose(separable_result, full_result))

### Zadaci

1. Provjerite da je i jezgra filtra srednje vrijednosti separabilna tako da je zapišete kao vanjski produkt dviju jednodimenzionalnih jezgri s jednakim koeficijentima.
2. Prethodna jezgra `sobel_x` nije simetrična. Izračunajte i konvoluciju i korelaciju slike `photo_like` s tom jezgrom te opišite kako se izlazi razlikuju. (Pomoć: razmislite na koji smjer promjene intenziteta reagira svaki postupak.)
3. Usporedite vrijeme izvođenja funkcije `convolve2d` s funkcijom `cv2.filter2D` na slici veličine 300 × 300 s Gaussovom jezgrom veličine 15 × 15, koristeći `%timeit`. Zatim usporedite vrijeme separabilne inačice s dva prolaza s punom dvodimenzionalnom inačicom. Koliko ubrzanje daje svaki pristup?
